In [1]:
# Import series of helper function for the notebook
from Helper_functions import create_tensorboard_callback, plot_loss_curves, compare_historys

## Get a text dataset

The dataset we're going to be using is Kaggle's introduction to NLP dataset (text samples of Tweets labelled as disaster or not disaster)

In [2]:
import pandas as pd

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

train_data.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [3]:
train_data['text'][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [4]:
# Shuffle training dataframe
train_data_shuffled = train_data.sample(frac=1, random_state=42)
train_data_shuffled

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0
...,...,...,...,...,...
5226,7470,obliteration,Merica!,@Eganator2000 There aren't many Obliteration s...,0
5390,7691,panic,NaN,just had a panic attack bc I don't have enough...,0
860,1242,blood,NaN,Omron HEM-712C Automatic Blood Pressure Monito...,0
7603,10862,NaN,NaN,Officials say a quarantine is in place at an A...,1


In [5]:
# Test data
test_data.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [6]:
# Count of samples
train_data.target.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

In [7]:
# Let's visualize some random training examples
import random 
random_index = random.randint(0, len(train_data)-5) # Create random indexes
for row in train_data_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
    _, text, target = row
    print(f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)")
    print(f"Text:\n{text}\n")
    print("---\n")

Target: 1 (real disaster)
Text:
Rly tragedy in MP: Some live to recount horror http://t.co/TTb9oiL8R2 #TopStories #India timesofindia

---

Target: 0 (not real disaster)
Text:
I liked a @YouTube video http://t.co/jK7nPdpWRo J. Cole - Fire Squad (2014 Forest Hills Drive)

---

Target: 1 (real disaster)
Text:
Leader of #Zionism STOP BURNING #Babies  https://t.co/6xYsDN2Xz0

---

Target: 0 (not real disaster)
Text:
Reddit Will Now Quarantine OffensiveåÊContent https://t.co/MjbIUvbMo6 http://t.co/I5cdTD8ftj

---

Target: 0 (not real disaster)
Text:
#hot  Reddit's new content policy goes into effect many horrible subreddits banned or quarantined http://t.co/algtcN8baf #prebreak #best

---



### Split data into training and validation sets
Since the test set has no labels and we need a way to evaluate our trained models, we'll split off some of the trianing data and create validation set.

In [8]:
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_data_shuffled["text"].to_numpy(),
                                                                            train_data_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)

In [9]:
# checking the length
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

In [10]:
# first 10 training sentences and their labels
train_sentences[:10], train_labels

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

## Converting text into numbers

There are two main techniques to convert text into numbers:

1. Text tokenization
2. Text embeddings

### Text tokenization

In [11]:
import tensorflow as tf 
from tensorflow.keras.layers import TextVectorization

text_vectorizer = TextVectorization(max_tokens=None, # how many words in the vocabulary (all of the different words in text)
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode="int",
                                    output_sequence_length=None)

In [12]:
# Find average number of tokens (words) in training tweets
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [13]:
# Setup text vectorization variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [14]:
# Fit the text vectorizer to the training text
text_vectorizer.adapt(train_sentences)

In [15]:
# Create a sample sentence and tokenize it 
sample_sentence = "There's a flood in my street!"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[264,   3, 232,   4,  13, 698,   0,   0,   0,   0,   0,   0,   0,
          0,   0]], dtype=int64)>

In [16]:
# Choose a random sentence from the training dataset and tokenize it
random_sentence = random.choice(train_sentences)
print(f"Original text:\n {random_sentence}\
    \n\nVectorized version:")
text_vectorizer([random_sentence])

Original text:
 First night with retainers in. It's quite weird. Better get used to it; I have to wear them every single night for the next year at least.    

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[  97,  285,   14, 9131,    4,   37, 1612, 1736,  441,   52,  493,
           5,   15,    8,   24]], dtype=int64)>

In [17]:
# Get the unique words in the vocabulary
words_in_vocab = text_vectorizer.get_vocabulary() # Get all of the unique words in our training data
top_5_words = words_in_vocab[:5] # get the most common words
bottom_5_words = words_in_vocab[-5:] # get the least common words
print(f"Number of words in vocab: {len(words_in_vocab)}")
print(f"5 most common words: {top_5_words}")
print(f"5 least common words: {bottom_5_words}")

Number of words in vocab: 10000
5 most common words: ['', '[UNK]', 'the', 'a', 'in']
5 least common words: ['pages', 'paeds', 'pads', 'padres', 'paddytomlinson1']


## Creating an Embedding using an Embedding Layer

The parameters we care most about for our embedding layer:
* `input_dim` = the size of our vocabulary
* `output_dim` = the size of the output embedding vector, for example, a value of 100 would mean each token gets represented by a vector 100 long
* `input_length` = length of the sequences being passed to the embedding layer

In [18]:
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length, # set input shape
                             output_dim=128,
                             input_length=max_length # how long is each input
                             )
embedding

In [19]:
# Get a random sentence from the training set
random_sentence = random.choice(train_sentences)
print(f"Original text:\n {random_sentence}\
    \n\nEmbedded version:")

# Embed the random sentence (turn it into dense vectors of fixed size)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original text:
 CommoditiesåÊAre Crashing Like It's 2008 All Over Again http://t.co/nJD1N5TxXe via @business    

Embedded version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.02188488, -0.00685988, -0.00014167, ..., -0.02139012,
         -0.01243738,  0.00795435],
        [ 0.03111954,  0.04093102,  0.02471754, ..., -0.03432226,
          0.02420704,  0.04257054],
        [ 0.0240276 , -0.03701005,  0.01710987, ...,  0.01559255,
         -0.03862375, -0.04697405],
        ...,
        [-0.00438427,  0.01924281,  0.0193075 , ..., -0.04464134,
         -0.0412056 ,  0.00614706],
        [-0.00438427,  0.01924281,  0.0193075 , ..., -0.04464134,
         -0.0412056 ,  0.00614706],
        [-0.00438427,  0.01924281,  0.0193075 , ..., -0.04464134,
         -0.0412056 ,  0.00614706]]], dtype=float32)>

In [20]:
# Check out a single token's embdedding
sample_embed[0][0], sample_embed[0][0].shape, random_sentence

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([ 0.02188488, -0.00685988, -0.00014167,  0.04599205,  0.01446483,
         0.03082964,  0.03499292, -0.01770835, -0.01889065,  0.01848209,
         0.02452486, -0.0189973 ,  0.01796501, -0.03798269, -0.04150502,
        -0.00409795,  0.03683784, -0.04819328, -0.04484671,  0.00591952,
         0.02363114,  0.0146015 , -0.02350372,  0.00323017,  0.00782074,
        -0.01591194,  0.02561761, -0.03929133,  0.0275849 , -0.00078647,
        -0.03070669, -0.01154318,  0.01926525,  0.02416705,  0.01052518,
         0.01640098, -0.02631066,  0.02937949, -0.04094739, -0.04301703,
        -0.03049148,  0.04513657, -0.03093116,  0.00548861,  0.03519512,
        -0.02752787,  0.02210606, -0.03582614, -0.00511041, -0.00973181,
        -0.01541607, -0.04599087,  0.03833962, -0.00906898, -0.02615796,
         0.04820717,  0.02627255,  0.0330786 , -0.0118608 ,  0.01495476,
        -0.03633649,  0.02529467, -0.04592416,  0.00179672, -0.02619686,
  

## Modelling a text dataset (running a series of experiments)

Now we've a got way to turn our text sequences into numbers, it's time to start building a series of modelling experiments.

We'll start with a baseline and move on from there

* Model 0: Naive Bayes (baseline).
* Model 1: Feed-Forward neural network (dense model)
* Model 2: LSTM model (RNN)
* Model 3: GRU model (RNN)\
* Model 4: Bidirectional-LSTM model (RNN)
* Model 5: 1D Convolutional Neural Network (CNN)
* Model 6: TensorFlow Hub pretrained feature extractor (using transfer learning for NLP)
* Model 7: Same as model 6 with 10% of training data.

### Model 0: Getting a baseline

As with all machine learning modelling experiments, it's important to create as baseline model so you have got a benchmark for future experiments to build upon.

To create our baseline, we'll use sklearn's multinomial naive bayes using the TF-IDF formula to convert our words to numbers.

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create tokenization and modelling pipeline
model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()), # convert words to numbers 
    ("clf", MultinomialNB()) # model the text
])

# Fit the pipeline to the training data
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [22]:
# Evaluate our baseline model
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

Our baseline model achieves an accuracy of: 79.27%


In [23]:
# Make predictions 
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
      dtype=int64)

### Creating an evaluation function for our model experiments

In [24]:
# Function to evaluate: accuracy, precision, recall and f1-score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
    """
    Calculates model accuracy, precision, recall and f1 score of a binary classification model.

    """
    # Calculate model accuracy 
    model_accuracy = accuracy_score(y_true, y_pred) * 100
    # Calculate model precision , recall and f1-score using "weighted" average
    model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
    model_results = {"accuracy": model_accuracy,
                     "precision": model_precision,
                     "recall": model_recall,
                     "f1": model_f1}
    return model_results    

In [25]:
# Get baseline result
baseline_results = calculate_results(y_true=val_labels,
                                     y_pred=baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

### Model 1: A simple dense model

In [26]:
# BUild model with the Funcitonal API
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype=tf.string) # inputs are 1-dimensional strings
x = text_vectorizer(inputs) # turn the input text into numbers
x = embedding(x) # embedding of the numberized inputs
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x) # output layers 
model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")

In [27]:
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [28]:
# Compile model
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [29]:
val_sentences.shape, val_labels.shape

((762,), (762,))

In [30]:
# fit the model
model_1_history = model_1.fit(x=train_sentences,
                              y=train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5

215/215 [==============================] - 6s 23ms/step - loss: 0.6095 - accuracy: 0.6895 - val_loss: 0.5350 - val_accuracy: 0.7585
Epoch 2/5
215/215 [==============================] - 4s 20ms/step - loss: 0.4416 - accuracy: 0.8155 - val_loss: 0.4681 - val_accuracy: 0.7848
Epoch 3/5
215/215 [==============================] - 4s 20ms/step - loss: 0.3458 - accuracy: 0.8608 - val_loss: 0.4669 - val_accuracy: 0.7900
Epoch 4/5
215/215 [==============================] - 4s 20ms/step - loss: 0.2833 - accuracy: 0.8921 - val_loss: 0.4639 - val_accuracy: 0.7900
Epoch 5/5
215/215 [==============================] - 4s 19ms/step - loss: 0.2370 - accuracy: 0.9120 - val_loss: 0.4789 - val_accuracy: 0.7756


In [31]:
# Check the results
model_1.evaluate(val_sentences, val_labels)

24/24 [==============================] - 0s 2ms/step - loss: 0.4789 - accuracy: 0.7756


[0.4789372682571411, 0.7755905389785767]

In [32]:
# Make some predictions and evalutate those
model_1_pred_probs = model_1.predict(val_sentences)
model_1_pred_probs.shape

24/24 [==============================] - 0s 2ms/step


(762, 1)

In [33]:
model_1_pred_probs[0]

array([0.37295768], dtype=float32)

In [34]:
# Convert model prediciton probabilities  to label format
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 1.], dtype=float32)>

In [35]:
# Calculate our model_1 results
model_1_results = calculate_results(y_true=val_labels,
                                    y_pred=model_1_preds)
model_1_results

{'accuracy': 77.55905511811024,
 'precision': 0.7786932114612464,
 'recall': 0.7755905511811023,
 'f1': 0.7729519699688302}

## Visualizing learned mebeddings

In [36]:
# Get the vocabulary from the text vectorization layer
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

(10000, ['', '[UNK]', 'the', 'a', 'in', 'to', 'of', 'and', 'i', 'is'])

In [37]:
# model 1 summary
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [38]:
# Get tghe weight matrix of embedding layer
# (these are the numerical representations of each token in our training, which have been learned for ~5 epochs)
embed_weights = model_1.get_layer("embedding").get_weights()[0]
embed_weights.shape

(10000, 128)

In [39]:
# Create mebedding files (we got this from TensorFlow's word embedding documnetation)
import io
out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(words_in_vocab):
    if index == 0:
        continue # Skip 0, it's padding.
    vec = embed_weights[index]
    out_v.write('\t'.join([str(x) for x in vec]) + "\n")
    out_m.write(word + "\n")
out_v.close()
out_m.close()

## Recurrent Neural Networks (RNN's)

RNN's are useful for sequential data.

The premise of a recurrent neural network is to use the representation of a previous input to aid the representation of a later input.


### Model 2: LSTM

Our structure of an RNN typically looks like this:

```
Input (txt) -> Tokenize -> Embedding -> Layers (RNNs/Dense) -> Output (label probability)
```

In [40]:
# Create an LSTM model
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
# print(x.shape)
x = layers.LSTM(64, return_sequences=True)(x) # when you're stacking RNN cells together , you need to set return_sequences=True
# print(x.shape)
x = layers.LSTM(64)(x)
# print(x.shape)
# x = layers.Dense(64, activation="relu")(x)
# print(x.shape)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

In [41]:
# Get a summary()
model_2.summary()

Model: "model_2_LSTM"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 lstm (LSTM)                 (None, 15, 64)            49408     
                                                                 
 lstm_1 (LSTM)               (None, 64)                33024     
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                      

In [42]:
# Compile the model
model_2.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [43]:
# Fit the model
model_2.history = model_2.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 [==============================] - 11s 31ms/step - loss: 0.2157 - accuracy: 0.9232 - val_loss: 0.5088 - val_accuracy: 0.7808
Epoch 2/5
215/215 [==============================] - 5s 25ms/step - loss: 0.1586 - accuracy: 0.9410 - val_loss: 0.5787 - val_accuracy: 0.7756
Epoch 3/5
215/215 [==============================] - 5s 24ms/step - loss: 0.1293 - accuracy: 0.9530 - val_loss: 0.6713 - val_accuracy: 0.7743
Epoch 4/5
215/215 [==============================] - 6s 26ms/step - loss: 0.1031 - accuracy: 0.9623 - val_loss: 0.7192 - val_accuracy: 0.7782
Epoch 5/5
215/215 [==============================] - 5s 25ms/step - loss: 0.0763 - accuracy: 0.9691 - val_loss: 1.0392 - val_accuracy: 0.7690


In [44]:
# Make predictions with LSTM model
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs[:10]

24/24 [==============================] - 1s 6ms/step


array([[1.4376630e-03],
       [4.0851489e-01],
       [9.9987221e-01],
       [5.3246032e-02],
       [2.4814776e-04],
       [9.9848229e-01],
       [9.8924249e-01],
       [9.9986398e-01],
       [9.9978018e-01],
       [5.3157073e-01]], dtype=float32)

In [45]:
# Convert model 2 pred probs to labels
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 0., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [46]:
# Calculate model 2 results
model_2_results = calculate_results(y_true=val_labels,
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 76.9028871391076,
 'precision': 0.7703609632684677,
 'recall': 0.7690288713910761,
 'f1': 0.7670626202962317}

In [47]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

### Model 3: GRU

Another popular and effective RNN component is the GRU or gated recurrent unit.
The GRU cell has similar features to an LSTM cell but but has less parameters.

In [48]:
# Buld an RNN using the GRU cell
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.GRU(64)(x)
# x = layers.GRU(64, return_sequences=True)(x)
# x = layers.LSTM(64, return_sequences=True)(x)
# x = layers.GRU(64, return_sequences=True)(x)
# x = layers.Dense(64, activation="relu")(x)
# x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_3 = tf.keras.Model(inputs, outputs, name="model_3_GRU")

In [49]:
model_3.summary()

Model: "model_3_GRU"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 gru (GRU)                   (None, 64)                37248     
                                                                 
 dense_2 (Dense)             (None, 1)                 65        
                                                                 
Total params: 1317313 (5.03 MB)
Trainable params: 1317313 (5.03 MB)
Non-trainable params: 0 (0.00 Byte)
_________________

In [50]:
# Compile the model
model_3.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [51]:
# Fit the model
model_3_history = model_3.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 [==============================] - 9s 26ms/step - loss: 0.1566 - accuracy: 0.9378 - val_loss: 0.6947 - val_accuracy: 0.7782
Epoch 2/5
215/215 [==============================] - 5s 21ms/step - loss: 0.0835 - accuracy: 0.9714 - val_loss: 0.8548 - val_accuracy: 0.7756
Epoch 3/5
215/215 [==============================] - 4s 21ms/step - loss: 0.0707 - accuracy: 0.9733 - val_loss: 1.0610 - val_accuracy: 0.7690
Epoch 4/5
215/215 [==============================] - 5s 21ms/step - loss: 0.0609 - accuracy: 0.9745 - val_loss: 1.0882 - val_accuracy: 0.7782
Epoch 5/5
215/215 [==============================] - 5s 21ms/step - loss: 0.0509 - accuracy: 0.9780 - val_loss: 1.1108 - val_accuracy: 0.7769


In [52]:
# Make some predictions with our GRU model
model_3_pred_probs = model_3.predict(val_sentences)
model_3_pred_probs[:10]

24/24 [==============================] - 1s 3ms/step


array([[6.9151749e-03],
       [7.1886206e-01],
       [9.9987251e-01],
       [9.2095040e-02],
       [7.0489616e-05],
       [9.9968433e-01],
       [9.6893835e-01],
       [9.9994040e-01],
       [9.9985266e-01],
       [5.2739090e-01]], dtype=float32)

In [53]:
# Convert model 3 pred probs to labels
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [54]:
# calculate model 3 results
model_3_results = calculate_results(y_true=val_labels,
                                    y_pred=model_3_preds)
model_3_results

{'accuracy': 77.69028871391076,
 'precision': 0.7781874595396938,
 'recall': 0.7769028871391076,
 'f1': 0.7751243211017478}

### Model 4: Bidirectional RNN
Normal RNN's go from left to right (just like you'd read an English sentence) however, a bidirectional RNN goes left to right also.

In [55]:
# Build a bidirectional RNN in TensorFlow
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
# x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_4 = tf.keras.Model(inputs, outputs, name="model_4_bidirectional")

In [56]:
# Get a summary
model_4.summary()

Model: "model_4_bidirectional"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 bidirectional (Bidirection  (None, 128)               98816     
 al)                                                             
                                                                 
 dense_3 (Dense)             (None, 1)                 129       
                                                                 
Total params: 1378945 (5.26 MB)
Trainable par

In [57]:
# Compile model
model_4.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [58]:
# Fit the model
model_4_hisrtory = model_4.fit(train_sentences,
                               train_labels,
                               epochs=5,
                               validation_data=(val_sentences, val_labels)
                               )

Epoch 1/5
215/215 [==============================] - 10s 30ms/step - loss: 0.1014 - accuracy: 0.9718 - val_loss: 0.9661 - val_accuracy: 0.7703
Epoch 2/5
215/215 [==============================] - 5s 25ms/step - loss: 0.0526 - accuracy: 0.9769 - val_loss: 1.2371 - val_accuracy: 0.7664
Epoch 3/5
215/215 [==============================] - 5s 24ms/step - loss: 0.0445 - accuracy: 0.9791 - val_loss: 1.3198 - val_accuracy: 0.7703
Epoch 4/5
215/215 [==============================] - 5s 25ms/step - loss: 0.0418 - accuracy: 0.9812 - val_loss: 1.3742 - val_accuracy: 0.7730
Epoch 5/5
215/215 [==============================] - 4s 21ms/step - loss: 0.0416 - accuracy: 0.9790 - val_loss: 1.5199 - val_accuracy: 0.7730


In [59]:
# Make predicitons with our bidirectional model
model_4_pred_probs = model_4.predict(val_sentences) 
model_4_pred_probs[:10]

24/24 [==============================] - 1s 4ms/step


array([[2.1855149e-03],
       [6.6593826e-01],
       [9.9999058e-01],
       [6.8345189e-02],
       [2.2942884e-05],
       [9.9993926e-01],
       [8.8437951e-01],
       [9.9999493e-01],
       [9.9999082e-01],
       [9.1925365e-01]], dtype=float32)

In [60]:
# Convert pred probs to pred labels
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [61]:
# Calculate the results of our bidirectional model
model_4_results = calculate_results(y_true=val_labels,
                                    y_pred=model_4_preds)
model_4_results

{'accuracy': 77.29658792650919,
 'precision': 0.7759760952910788,
 'recall': 0.7729658792650919,
 'f1': 0.7702964374538457}

### Convolution Neural Networks for Text (and other types of sequences)

We've used CNNs for images are typically 2D (height x width)... however, our text data is 1D.
Previously we've Conv2D for our image data but now we're going to use Conv1D.
The typical structure of a COnv1D model for sequences (in our case text)

```
Inputs (text) -> Tokenization -> Embedding -> Layer(s) (typically Conv1D + pooling) -> Outputs (class probabilities)
```

### Model 5: Conv1D

In [64]:
# Test out our embedding layer, Con1D layer and max pooling 
from tensorflow.keras import layers
embedding_test = embedding(text_vectorizer(["this is a test sentence"]))
conv_1d = layers.Conv1D(filters=32,
                        kernel_size=5,
                        strides=1, # default  
                        activation="relu",
                        padding="same")
conv_1d_output = conv_1d(embedding_test) # pass test embedding through conv1d layer
max_pool = layers.GlobalMaxPool1D()
max_pool_output = max_pool(conv_1d_output) 
embedding_test.shape, conv_1d_output.shape, max_pool_output.shape

(TensorShape([1, 15, 128]), TensorShape([1, 15, 32]), TensorShape([1, 32]))

In [66]:
# Create 1-dimensional convolutional layer to model sequence
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.Conv1D(filters=64, kernel_size=5, strides=1, activation="relu", padding="valid")(x)
x = layers.GlobalMaxPool1D()(x)
# x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_5 = tf.keras.Model(inputs, outputs, name="model_5_Conv1D")

# Compile Conv1D
model_5.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

# Get a summary of our Conv1D model
model_5.summary()

Model: "model_5_Conv1D"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 conv1d_3 (Conv1D)           (None, 11, 64)            41024     
                                                                 
 global_max_pooling1d_3 (Gl  (None, 64)                0         
 obalMaxPooling1D)                                               
                                                                 
 dense_5 (Dense)             (None, 1)              

In [67]:
#  Fit the model
model_5_history = model_5.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 [==============================] - 7s 29ms/step - loss: 0.1316 - accuracy: 0.9545 - val_loss: 0.9026 - val_accuracy: 0.7690
Epoch 2/5
215/215 [==============================] - 6s 27ms/step - loss: 0.0760 - accuracy: 0.9717 - val_loss: 1.0866 - val_accuracy: 0.7703
Epoch 3/5
215/215 [==============================] - 6s 28ms/step - loss: 0.0618 - accuracy: 0.9752 - val_loss: 1.1331 - val_accuracy: 0.7612
Epoch 4/5
215/215 [==============================] - 6s 27ms/step - loss: 0.0543 - accuracy: 0.9780 - val_loss: 1.2156 - val_accuracy: 0.7598
Epoch 5/5
215/215 [==============================] - 6s 29ms/step - loss: 0.0509 - accuracy: 0.9791 - val_loss: 1.2701 - val_accuracy: 0.7598


In [68]:
# Make some predictions with our Conv1D model
model_5_pred_probs = model_5.predict(val_sentences)
model_5_pred_probs[:10]

24/24 [==============================] - 0s 2ms/step


array([[9.18182954e-02],
       [8.07801306e-01],
       [9.99903679e-01],
       [1.00185476e-01],
       [1.85700713e-07],
       [9.95579004e-01],
       [9.63648498e-01],
       [9.99993384e-01],
       [9.99999523e-01],
       [8.97659957e-01]], dtype=float32)

In [69]:
# Convert model 5 pred probs to labels
model_5_preds = tf.squeeze(tf.round(model_5_pred_probs))
model_5_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [71]:
# Evaluate model 5 predicitons
model_5_results = calculate_results(y_true=val_labels,
                                    y_pred=model_5_preds)
model_5_results

{'accuracy': 75.98425196850394,
 'precision': 0.7597625353264251,
 'recall': 0.7598425196850394,
 'f1': 0.7586850461611226}

In [72]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}